In [10]:
from pathlib import Path
import re
import pandas as pd
import os

curfolder = os.getcwd()
trialdata = os.path.join(curfolder, "data", "Data_processed", "Data_trials")

# Collect basenames (no extension) → paths
name_to_paths = {}
for p in Path(trialdata).rglob("*"):
    if p.is_file():
        name_to_paths.setdefault(p.stem, []).append(str(p))

name_set = set(name_to_paths.keys())

# Updated regex:
# - allows any nominal_srate number: srate\d+
# - makes the camera piece optional: (?:_c\d+)? 
# Examples matched:
#   20_1_trial_52_MyWebcamFrameStream_nominal_srate500_p1_dik_gebaren_cut1
#   11_2_trial_35_MyWebcamFrameStream_nominal_srate500_p1_hoog_geluiden_c1_cut18
webcam_re = re.compile(
    r"""^(?P<prefix>.*?_trial_\d+)_
        MyWebcamFrameStream_
        nominal_srate\d+_
        (?P<suffix>p\d+_.+?(?:_c\d+)?_cut\d+)$
    """,
    re.VERBOSE
)

rows = []
for name in name_set:
    m = webcam_re.match(name)
    if not m:
        continue
    prefix = m.group("prefix")
    suffix = m.group("suffix")

    # Expected video (still sans extension)
    expected_video = f"{prefix}_{suffix}_video_raw"

    rows.append({
        "webcam_name": name,
        "webcam_path(s)": " | ".join(name_to_paths.get(name, [])),
        "expected_video_name": expected_video,
        "video_found": expected_video in name_set,
        "video_path(s)": " | ".join(name_to_paths.get(expected_video, [])),
    })

df_webcam = pd.DataFrame(rows).sort_values("webcam_name").reset_index(drop=True)
df_webcam

# show False video_found entries
df_webcam_missing = df_webcam[df_webcam["video_found"] == False]
display(df_webcam_missing)



,webcam_name,webcam_path(s),expected_video_name,video_found,video_path(s)


# audio check

In [11]:
from pathlib import Path
import re
import pandas as pd
import os

# Root folder to scan (current working dir by default)
root = Path(trialdata)

# Which sample rates to require for each extension
REQUIRED = {
    "wav": {48000, 16000},
    "csv": {16000},  # adjust to {16000, 48000} if you also expect csv at 48k
}

# Regex to parse filenames like (both are accepted):
# 1) 1_1_trial_16_Mic_nominal_srate48000_p1_kotsen_gebaren_cut1.wav
# 2) 1_1_trial_16_nominal_srate16000_p1_kotsen_gebaren_cut1.csv   (no Mic_)
# Also tolerates an optional _c<digit> before _cutN (if it ever occurs)
audio_re = re.compile(
    r"""^(?P<prefix>.*?_trial_\d+)_               # e.g., 1_1_trial_16_
         (?:(?:Mic)_)?                            # optional 'Mic_'
         nominal_srate(?P<rate>\d+)_              # sample rate number
         (?P<suffix>p\d+_.+?(?:_c\d+)?_cut\d+)    # p1_...(_c1)?_cutN
         \.(?P<ext>wav|csv)$                      # extension
     """,
    re.IGNORECASE | re.VERBOSE,
)

# Collect matches
records = []
for p in root.rglob("*"):
    if not p.is_file():
        continue
    name = p.name
    m = audio_re.match(name)
    if not m:
        continue

    prefix = m.group("prefix")
    rate = int(m.group("rate"))
    suffix = m.group("suffix")
    ext = m.group("ext").lower()

    # "key" groups files that should correspond (ignoring sample rate & extension)
    key = f"{prefix}_{suffix}"

    records.append({
        "key": key,
        "rate": rate,
        "ext": ext,
        "path": str(p),
        "name": name,
    })

df_all = pd.DataFrame(records)

# If nothing matched, show an empty result with a hint
if df_all.empty:
    print("No audio files matched the expected pattern. Check names and the regex.")
    display(pd.DataFrame(columns=[
        "key","wav_48000","wav_16000","csv_16000","wav_48000_paths","wav_16000_paths","csv_16000_paths"
    ]))
else:
    # Build availability flags and collect paths per key
    def paths_for(k, e, r):
        return df_all[(df_all["key"] == k) & (df_all["ext"] == e) & (df_all["rate"] == r)]["path"].tolist()

    keys = sorted(df_all["key"].unique())

    rows = []
    for k in keys:
        row = {"key": k}

        # WAVs
        for req_rate in sorted(REQUIRED["wav"]):
            col = f"wav_{req_rate}"
            col_paths = f"{col}_paths"
            ps = paths_for(k, "wav", req_rate)
            row[col] = len(ps) > 0
            row[col_paths] = " | ".join(ps)

        # CSVs
        for req_rate in sorted(REQUIRED["csv"]):
            col = f"csv_{req_rate}"
            col_paths = f"{col}_paths"
            ps = paths_for(k, "csv", req_rate)
            row[col] = len(ps) > 0
            row[col_paths] = " | ".join(ps)

        # Overall completeness
        row["all_required_present"] = all(
            row.get(f"wav_{r}", False) for r in REQUIRED["wav"]
        ) and all(
            row.get(f"csv_{r}", False) for r in REQUIRED["csv"]
        )

        rows.append(row)

    df = pd.DataFrame(rows)

    # Nice column order
    cols = ["key"]
    cols += [f"wav_{r}" for r in sorted(REQUIRED["wav"])]
    cols += [f"csv_{r}" for r in sorted(REQUIRED["csv"])]
    cols += [f"{c}_paths" for c in cols[1:]] + ["all_required_present"]
    df = df[cols]

    # Show the full table
    df

display(df[ df["all_required_present"] == False ])

,key,wav_16000,wav_48000,csv_16000,wav_16000_paths,wav_48000_paths,csv_16000_paths,all_required_present
154,15_1_trial_20_p0_niet_geluiden_cut4,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
155,15_1_trial_22_p0_verbranden_geluiden_cut1,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
156,15_1_trial_29_p1_vogel_geluiden_cut1,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
157,15_1_trial_31_p1_kauwen_geluiden_cut1,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
158,15_1_trial_40_p0_slang_combinatie_cut18,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
159,15_1_trial_42_p0_bijten_combinatie_cut1,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
160,15_1_trial_6_p0_onweer_gebaren_cut18,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
161,15_1_trial_7_p0_slaan_gebaren_cut18,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
162,15_1_trial_8_p0_kotsen_gebaren_cut1,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False
201,16_2_trial_100_p1_sterk_geluiden_c1_cut18,True,False,True,f:\flesh_data_processed\01_XDF_processing\data...,,f:\flesh_data_processed\01_XDF_processing\data...,False


15 and 16 are missing 48 kHz audio so this is ok